# `operator_gb` — an interactive tutorial

**NOTE:** This notebook can be run online, without installing anything, via Binder:

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ClemensHofstadler/operator_gb/HEAD?labpath=Tutorial.ipynb)

The Binder session comes with SageMath and a preinstalled version of the package `operator_gb`, so all cells below can be executed and modified directly in the browser.

Cells are executed with `Shift`+`Enter`. Let us start by loading the package.

In [ ]:
from operator_gb import *

# The Moore-Penrose inverse

Recall that the Moore-Penrose inverse of a complex matrix $A$ (or more generally of a linear operator $A$) is the complex matrix (resp. the linear operator) $B$ satisfying
$$
ABA = A, \qquad BAB = B, \qquad AB = B^\ast A^\ast, \qquad BA = A^\ast B^\ast,
$$
where $P^\ast$ denotes the Hermitian adjoint of a complex matrix (resp. a linear operator) $P$. If it exists, it is unique, and it is denoted by $A^\dagger$.

## How the package proves such statements

Every operator identity $P = Q$ is translated into the noncommutative polynomial $P - Q$, with one indeterminate for every basic operator. A claimed identity then follows from the assumed identities if the polynomial of the claim lies in the ideal generated by the polynomials of the assumptions.

The command `certify` tries to verify such an ideal membership. If it succeeds, it produces a **cofactor representation**
$$
  f = \sum_{i} p_i \, g_i \, q_i
$$
of the claim $f$ in terms of the assumptions $g_i$. This is a certificate that can be checked independently.

# Part 1: Three worked examples

## Uniqueness of the Moore-Penrose inverse
*(Fact 1 in [Hog13, Sec. I.5.7])*

If $B$ and $C$ both satisfy the Moore-Penrose identities for $A$, then $B = C$.

**Required property:** Encoding the Hermitian adjoint $^\ast$

**Strategy:** Add for every assumed identity $P = Q$ also the corresponding adjoint identity $P^\ast = Q^\ast$, and simplify all operator expressions using the following identities before translating them into polynomials:
$$
(P+Q)^\ast = P^\ast + Q^\ast, \qquad\qquad (PQ)^\ast = Q^\ast P^\ast, \qquad\qquad (P^\ast)^\ast = P.
$$

The adjoint of a basic operator is a separate indeterminate, by convention named `a_adj`. The command `pinv` generates the four Moore-Penrose identities, and `add_adj` adds the adjoint of every assumption.

In [ ]:
# first create the FreeAlgebra containing all indeterminates
F.<a, b, c, a_adj, b_adj, c_adj> = FreeAlgebra(QQ)

# assumptions and claim are ordinary SageMath noncommutative polynomials
# the command pinv generates the polynomials of the Moore-Penrose identities
# the syntax for this command is pinv(var, name_of_the_MP_inverse, var_adj, name_of_the_MP_adj)
Pinv_B = pinv(a, b, a_adj, b_adj)
Pinv_C = pinv(a, c, a_adj, c_adj)
# the command add_adj adds the corresponding adjoint identities
assumptions = add_adj(Pinv_B + Pinv_C)
claim = b - c

print("The assumptions are %s.\n" % str(assumptions))
print("The claim is %s.\n" % str(claim))

# call certify
proof = certify(assumptions, claim)

print("The proof is:")
pretty_print_proof(proof, assumptions)

### Reading the output

The certificate is a list of triples $(p_i, i, q_i)$, meaning "multiply the $i$-th assumption by $p_i$ from the left and by $q_i$ from the right". Two commands help to work with it:

- `pretty_print_proof(proof, assumptions)` prints the certificate as a readable identity — this is what was used above;
- `expand_cofactors(proof, assumptions)` multiplies the certificate out again, which verifies the proof.

In [ ]:
print("The certificate consists of %d steps.\n" % len(proof))

# every step is a triple (left cofactor, index of the assumption, right cofactor)
print("The first step is %s, i.e., it uses the assumption %s.\n"
      % (str(proof[0]), str(assumptions[proof[0].i()])))

# multiplying the certificate out again has to give back the claim
expand_cofactors(proof, assumptions)

## Existence of the Moore-Penrose inverse
*(Fact 1 in [Hog13, Sec. I.5.7])*

Every complex matrix has a Moore-Penrose inverse.

**Required property:** Proving an existential statement

**Strategy:** Construct an explicit expression for the existentially quantified variable.

To do this, the package provides dedicated methods, such as the command `find_equivalent_expression`. Here we start from the fact that, for every matrix $A$, there are matrices $B$ and $C$ with $A = BA^\ast A$ and $A = AA^\ast C$. We introduce a dummy indeterminate $X$ satisfying the Moore-Penrose identities for $A$, and then search for an expression that is equal to $X$ but written in terms of $A$, $B$, $C$ and their adjoints.

In [ ]:
F.<a,b,c,a_adj,b_adj,c_adj,x,x_adj> = FreeAlgebra(QQ)

# in this example, we first have to find an expression for the Moore-Penrose inverse
# we do this using our heuristics
# to this end, we introduce a dummy variable x satisfying the Moore-Penrose equations
# and then search for an expression equivalent to x but in terms of a,b,c and their adjoints
assumptions = add_adj([a - b*a_adj*a, a - a*a_adj*c] + pinv(a, x, a_adj, x_adj))
I = NCIdeal(assumptions)
candidates = I.find_equivalent_expression(x)
# one of the candidates shows that X = A^* C B^*
print("Found candidates for the Moore-Penrose inverse: %s\n" % str(candidates))

# we show that the candidate a_adj*c*b_adj is indeed the Moore-Penrose inverse of a
# by showing that it satisfies the four Moore-Penrose equations
MP_candidate = a_adj * c * b_adj
MP_candidate_adj = adj(MP_candidate)
claim = pinv(a, MP_candidate, a_adj, MP_candidate_adj)
print("The assumptions are %s\n" % str(assumptions))
print("The claim is %s\n" % str(claim))

# call certify
proof = certify(assumptions, claim)

print("The proofs consist of %s steps, respectively.\n" % str(list(map(len,proof))))

## The Moore-Penrose inverse is an involution
*(Fact 10 in [Hog13, Sec. I.5.7])*

$(A^\dagger)^\dagger = A$

**Required property:** Several Moore-Penrose inverses at once

**Strategy:** Every Moore-Penrose inverse occurring in a statement gets its own indeterminate and its own set of Moore-Penrose identities. Here `x` is the Moore-Penrose inverse of `a`, and `y` is the Moore-Penrose inverse of `x`; the claim is `a = y`.

In [ ]:
F.<a, x, y, a_adj, x_adj, y_adj> = FreeAlgebra(QQ)

# x is the Moore-Penrose inverse of a, and y is the Moore-Penrose inverse of x
Pinv_x = pinv(a, x, a_adj, x_adj)
Pinv_y = pinv(x, y, x_adj, y_adj)

assumptions = add_adj(Pinv_x + Pinv_y)
claim = a - y

print("The assumptions are %s\n" % str(assumptions))
print("The claim is %s\n" % str(claim))

proof = certify(assumptions, claim)
print("The proof consists of %d steps.\n" % len(proof))

pretty_print_proof(proof, assumptions)

# Part 2: Encoding properties of matrices and operators

The examples above only needed adjoints. Most statements about matrices and operators involve further properties — being invertible, having full rank, being real, an inclusion of ranges. All of these can be expressed by polynomial identities, sometimes after introducing auxiliary indeterminates. Below, we collect some useful encodings.

| Property | Encoded by |
| :--- | :--- |
| $A^\ast$, the adjoint of $A$ | a separate indeterminate `a_adj`; adjoin the adjoint of *every* assumption (`add_adj`) |
| $I$ is an identity matrix | `i*i - i`, `i - i_adj`, and `a*i - a`, `i*b - b` for all basic operators |
| $A$ has full row rank | `a*u - i` for a fresh indeterminate `u` and `i` modelling the identitiy matrix |
| $A$ has full column rank | `v*a - i` for a fresh indeterminate `v` and `i` modelling the identitiy matrix |
| $A$ is invertible | `a*a_inv - id2` and `a_inv*a - id1` and `id1`, `id2` modelling identitiy matrices |
| $A$ is real | `a - a_c`, with `a_c`, `a_tr` for the conjugate and the transpose of $A$ |
| $A$ is Hermitian | `a - a_adj` |
| $A$ is normal | `a*a_adj - a_adj*a` |
| $A$ is idempotent | `a*a - a` |
| $A$ has orthonormal columns | `a_adj*a - i` with `i` modelling the identitiy matrix |
| $\operatorname{range}(A) \subseteq \operatorname{range}(B)$ | `a - b*x` for a fresh indeterminate `x` |
| $\ker(A) \subseteq \ker(B)$ | assume `a*x`, claim `b*x`, for a fresh indeterminate `x` |

## 1. Adjoints

The adjoint $A^\ast$ of a basic operator $A$ is a new indeterminate, by convention named `a_adj`. Adjoints of composite expressions are then computed by the rules
$$
(P+Q)^\ast = P^\ast + Q^\ast, \qquad (PQ)^\ast = Q^\ast P^\ast, \qquad (P^\ast)^\ast = P,
$$
which is what the command `adj` does. Whenever an identity $P = Q$ is assumed, the identity $P^\ast = Q^\ast$ has to be assumed as well — this is what `add_adj` does for a whole list of assumptions.

In [ ]:
F.<a, b, a_adj, b_adj> = FreeAlgebra(QQ)

# adj computes the adjoint of an operator expression
print(adj(a*b))
print(adj(a_adj*b*a))

# add_adj takes a list of assumptions and adds the adjoint of each of them;
# here, the four Moore-Penrose identities generated by pinv
for f in add_adj(pinv(a, b, a_adj, b_adj)):
    print(f)

## 2. Quivers

When working with rectangular matrices, not every product and not every sum are defined. A *quiver* records this: its vertices are the underlying spaces, and every basic operator is an edge from its source to its target. Expressions are then only formed along composable paths.

A quiver is given as a list of triples `(source, target, label)`. Passing one to `certify` has the effect that all expressions are tested for compatibility with this quiver. If a non-compatible expression is encountered as part of the input (which means that some sum or product within that expression would not exist for the defined matrix formats), `certify` aborts with an error.

The command `Q.plot()` draws the quiver; note that it draws the vertices without their names.

In [ ]:
F.<a, a_adj, a_dag, a_dag_adj> = FreeAlgebra(QQ)

# A maps space 1 to space 2, hence A^* and A^dag map back, and (A^dag)^* forward again
Q = Quiver([(1, 2, u) for u in [a, a_dag_adj]] + [(2, 1, u) for u in [a_adj, a_dag]])
print(Q)

# a*a_dag*a - a is a meaningful expression, a*a is not
print("a*a_dag*a - a is compatible: %s" % str(Q.is_compatible(a*a_dag*a - a)))
print("a*a           is compatible: %s" % str(Q.is_compatible(a*a)))

# Q.plot() draws the quiver
Q.plot()

# the assumption is not compatible with the quiver, i.e., in terms of matrices this expression does not exist
# certify aborts with a ValueError
assumptions = [a*a]
claim = a*a_adj*a
certify(assumptions, claim, Q)

## 3. Identity matrices

As soon as the identity matrix of a space occurs in a statement, it needs its own indeterminate `i`, together with the identities it satisfies:

- `i*i - i` and `i - i_adj`, since it is an idempotent, Hermitian operator;
- `a*i - a` and `i*b - b` for every basic operator `a`, `b` for which these products are defined.

The second group is easy to forget, and forgetting it is the most common reason why a proof that "should" work does not.

Note that different spaces have different identity matrices: if $A$ maps space 1 to space 2, then $I_1$ and $I_2$ are two distinct indeterminates, with $AI_1 = A = I_2A$.

## 4. One-sided inverses: full rank, injectivity, surjectivity, invertibility

A matrix has full row rank exactly iff it has a right inverse, and full column rank exactly iff it has a left inverse. So full row rank of $A$ is encoded by
$$
AU = I,
$$
where $U$ is a fresh indeterminate satisfying no further hypotheses and $I$ is an identity matrix as in point 3. Analogously, full column rank of $A$ corresponds to $VA = I$, and an invertible $A$ satisfies both with the *same* $U = V$.

More generally, the same trick encodes surjectivity and injectivity of operators. Note that nothing at all is assumed about $U$ and $V$ beyond $AU = I$ (resp. $VA = I$).

## 5. Real matrices

The Hermitian adjoint decomposes into complex conjugation and transposition, $A^\ast = (A^C)^T$, and being real is then expressed by $A = A^C$.

In terms of polynomials, this means introducing two further indeterminates `a_c` and `a_tr` for every basic operator and adding the polynomial `a - a_c`. Additionally, for every assumption $P = Q$, the adjoint identity $P^\ast = Q^\ast$, the transposed identity $P^T = Q^T$, and the conjugated identity $P^C = Q^C$ have to be translated into polynomials as well. These additional identities first have to be simplified using the following rules, which relate the three function symbols to each other. Let $\alpha, \beta, \gamma, \delta \in \{\ast, C, T\}$ with $\gamma \neq \alpha, \beta$ and $\delta \neq C$. Then

- $(P+Q)^\alpha = P^\alpha + Q^\alpha$
- $(P^\alpha)^\beta = P$ if $\alpha = \beta$, and $(P^\alpha)^\beta = P^\gamma$ if $\alpha \neq \beta$
- $(PQ)^C = P^C Q^C$
- $(PQ)^\delta = Q^\delta P^\delta$

## 6. Special classes of matrices

Many properties are already polynomial identities and need no auxiliary indeterminates at all:

| $A$ is ... | polynomial |
| :--- | :--- |
| Hermitian, $A = A^\ast$ | `a - a_adj` |
| normal, $AA^\ast = A^\ast A$ | `a*a_adj - a_adj*a` |
| idempotent, $A^2 = A$ | `a*a - a` |
| an orthogonal projection | `a*a - a` and `a - a_adj` |
| a partial isometry, $A^\dagger = A^\ast$ | `a_dag - a_adj` |
| EP, $AA^\dagger = A^\dagger A$ | `a*a_dag - a_dag*a` |

Together with an identity matrix `i` as in point 3, orthonormal columns of $A$ become `a_adj*a - i` and orthonormal rows become `a*a_adj - i`.

## 7. Ranges and kernels

By a classical result called *Douglas' lemma*, an inclusion of ranges is equivalent to a factorisation:
$$
  \operatorname{range}(P) \subseteq \operatorname{range}(Q) \iff P = QX \text{ for some } X.
$$
So a range inclusion is *assumed* by adding the polynomial `p - q*x` for a fresh indeterminate `x`, and it is *proven* by finding an explicit $X$. The command `find_equivalent_expression` does the latter: `I.find_equivalent_expression(p, prefix=q)` searches the ideal for an element of the form $P - Q(\dots)$.

Kernel inclusions are even simpler. Since
$$
  \ker(P) \subseteq \ker(Q) \iff \bigl(Px = 0 \implies Qx = 0\bigr),
$$
one adds `p*x` to the assumptions, for a fresh indeterminate `x`, and claims `q*x`.

In [ ]:
F.<a, a_adj, a_dag, a_dag_adj> = FreeAlgebra(QQ)
I = NCIdeal(add_adj(pinv(a, a_dag, a_adj, a_dag_adj)))

# range(A) is contained in range(A A^dag), because A = A A^dag X for X = A
print(I.find_equivalent_expression(a, prefix=a*a_dag, heuristic='naive')[0])

## 8. Cancellability

Some proofs need a cancellation step that is not an algebraic consequence of the assumptions, for example
$$
  P^\ast P = 0 \implies P = 0,
$$
which holds for all matrices and, more generally, for all bounded linear operators, since $\lVert P \rVert^2 = \lVert P^\ast P\rVert$.

There are two ways to exploit this. Instead of proving an identity $P = Q$ directly, one can prove $(P-Q)^\ast(P-Q) = 0$ and conclude $P = Q$. Alternatively, `NCIdeal.apply_left_cancellability` and `NCIdeal.apply_right_cancellability` search the ideal for consequences of a cancellability assumption and return them as additional assumptions.

# Part 3: Try yourself

Below are ten theorems about the Moore-Penrose inverse, roughly in increasing order of difficulty. For each of them the statement is given, but the code is only partly given: everywhere you see `...`, something is missing.

A few reminders:

- `pinv(a, b, a_adj, b_adj)` returns the four Moore-Penrose identities saying that `b` is the Moore-Penrose inverse of `a`, where `a_adj` and `b_adj` are the adjoints of `a` and `b`;
- `add_adj(assumptions)` adds the adjoint of every assumption;
- `certify(assumptions, claim)` reports whether it could verify the claim, and returns the certificate. To encode rectangular matrices, one can also give a quiver `Q` as argument to `certify`: `certify(assumptions, claim, Q)` then also checks if all input expressions are compatible with `Q`.
- if `certify` fails, look first for a missing assumption — a forgotten identity-matrix relation, or a forgotten adjoint — and only then increase `maxiter`.

## Exercise 1: Hermitian idempotents
*(Fact 13 in [Hog13, Sec. I.5.7])*

If $A = A^\ast$ and $A = A^2$, then $A^\dagger = A$.

*Needed:* adjoints (point 1).

In [ ]:
F.<a, a_dag, a_adj, a_dag_adj> = FreeAlgebra(QQ)

Pinv_a = pinv(a, a_dag, a_adj, a_dag_adj)

# encode that A is Hermitian and that A is idempotent
a_hermitian = ...
a_idempotent = ...

assumptions = add_adj(Pinv_a + [a_hermitian, a_idempotent])
# state the claim A^dag = A
claim = ...

proof = certify(assumptions, claim)
print("The proof consists of %d steps." % len(proof))

## Exercise 2: The Moore-Penrose inverse of the adjoint
*(Fact 10 in [Hog13, Sec. I.5.7])*

$(A^\ast)^\dagger = (A^\dagger)^\ast$

*Needed:* a second set of Moore-Penrose identities, as in the involution example of Part 1. Here `x` denotes $(A^\ast)^\dagger$ and `x_adj` its adjoint. Mind the order of the arguments of `pinv`: the first argument is the operator, the second its Moore-Penrose inverse, and the last two are their adjoints.

In [ ]:
F.<a, a_dag, a_adj, a_dag_adj, x, x_adj> = FreeAlgebra(QQ)

# a_dag is the Moore-Penrose inverse of a
Pinv_a = pinv(a, a_dag, a_adj, a_dag_adj)
# x is the Moore-Penrose inverse of a_adj
Pinv_a_adj = pinv(...)

assumptions = add_adj(Pinv_a + Pinv_a_adj)
# state the claim (A^*)^dag = (A^dag)^*
claim = ...

# now we also add in a quiver to the game to encode that our matrices can be rectangular
# The quiver has two vertices (called 1 and 2) and one edge for each variable
Q = Quiver([(1, 2, u) for u in [a, a_dag_adj, x]] + [(2, 1, u) for u in [a_adj, a_dag, x_adj]])

proof = certify(assumptions, claim, Q)
print("The proof consists of %d steps." % len(proof))

## Exercise 3: Normal matrices commute with their Moore-Penrose inverse
*(Fact 15 in [Hog13, Sec. I.5.7])*

If $A$ is normal, i.e., $AA^\ast = A^\ast A$, then $AA^\dagger = A^\dagger A$.

*Needed:* adjoints (point 1). A normal matrix is square, so the quiver consists of only a single vertex.

In [ ]:
F.<a, a_dag, a_adj, a_dag_adj> = FreeAlgebra(QQ)

Pinv_a = pinv(a, a_dag, a_adj, a_dag_adj)

# encode that A is normal
a_normal = ...

assumptions = add_adj(Pinv_a + [a_normal])
# state the claim A A^dag = A^dag A
claim = ...

# trivial quiver with only one vertex (called 1) because all matrices must be square
Q = Quiver([(1, 1, u) for u in [a, a_adj, a_dag, a_dag_adj]])

proof = certify(assumptions, claim, Q)
print("The proof consists of %d steps." % len(proof))

## Exercise 4: Orthonormal columns or rows
*(Fact 12 in [Hog13, Sec. I.5.7])*

If $A$ has orthonormal columns or orthonormal rows, then $A^\dagger = A^\ast$.

*Needed:* identity matrices (point 3). Here `id1` is the identity on the source of $A$ and `id2` the identity on its target. This is really two statements, so there are two proofs to run — and note that the two cases share all their other assumptions.

In [ ]:
F.<a, a_dag, a_adj, a_dag_adj, id1, id1_adj, id2, id2_adj> = FreeAlgebra(QQ)

Pinv_a = pinv(a, a_dag, a_adj, a_dag_adj)

# id1 is the identity on the source of a, id2 the identity on its target;
# the first two relations say A*I1 = A = I2*A -- add the two analogous
# relations for a_dag, which maps in the opposite direction
identity_matrix = [a*id1 - a, id2*a - a, ..., ...]

# encode that A has orthonormal columns, resp. orthonormal rows
a_orthonormal_cols = ...
a_orthonormal_rows = ...

# state the claim A^dag = A^*
claim = ...

Q = Quiver([(1, 2, u) for u in [a, a_dag_adj]] +
           [(2, 1, u) for u in [a_adj, a_dag]] +
           [(1, 1, id1), (1, 1, id1_adj), (2, 2, id2), (2, 2, id2_adj)])

# case 1: orthonormal columns
assumptions_1 = add_adj(Pinv_a + identity_matrix + [a_orthonormal_cols])
proof = certify(assumptions_1, claim, Q)
print("The proof consists of %d steps.\n" % len(proof))

# case 2: orthonormal rows
assumptions_2 = add_adj(Pinv_a + identity_matrix + [a_orthonormal_rows])
proof = certify(assumptions_2, claim, Q)
print("The proof consists of %d steps." % len(proof))

## Exercise 5: Non-singular square matrices
*(Fact 11 in [Hog13, Sec. I.5.7])*

If $A$ is a non-singular square matrix, then $A^\dagger = A^{-1}$.

*Needed:* identity matrices (point 3) and a two-sided inverse (point 4). All relations for the two identity matrices are already given — there are a lot of them, one pair for every basic operator. (Are all of them actually needed for the proof?)

In [ ]:
F.<a, a_dag, a_adj, a_dag_adj, a_inv, a_inv_adj, id1, id1_adj, id2, id2_adj> = FreeAlgebra(QQ)

Pinv_a = pinv(a, a_dag, a_adj, a_dag_adj)

# id1 is the identity on the source of a, id2 the identity on its target;
# every basic operator has to be multiplied by them from both sides
identity_matrices = [id2*a - a, a*id1 - a,
                     a_dag*id2 - a_dag, id1*a_dag - a_dag,
                     a_inv*id2 - a_inv, id1*a_inv - a_inv,
                     a_adj*id2 - a_adj, id1*a_adj - a_adj,
                     id2*a_dag_adj - a_dag_adj, a_dag_adj*id1 - a_dag_adj,
                     id2*a_inv_adj - a_inv_adj, a_inv_adj*id1 - a_inv_adj]

# encode that a_inv is a two-sided inverse of a -- mind which identity
# matrix appears on which side
a_inverse = ...

assumptions = add_adj(Pinv_a + a_inverse + identity_matrices)
# state the claim A^dag = A^(-1)
claim = ...

Q = Quiver([(1, 2, u) for u in [a, a_dag_adj, a_inv_adj]] +
           [(2, 1, u) for u in [a_adj, a_dag, a_inv]] +
           [(1, 1, id1), (1, 1, id1_adj), (2, 2, id2), (2, 2, id2_adj)])

proof = certify(assumptions, claim, Q)
print("The proof consists of %d steps." % len(proof))

## Exercise 6: The Moore-Penrose inverse via the Gram matrix
*(Fact 18 in [Hog13, Sec. I.5.7])*

It holds that $A^\dagger = (A^\ast A)^\dagger A^\ast$.

*Needed:* an auxiliary indeterminate `b` for the Gram matrix $A^\ast A$, together with its own Moore-Penrose identities. Note that the Gram matrix is Hermitian, so its adjoint is `b` itself — which is what you pass to `pinv` as the third argument.

In [ ]:
F.<a, a_adj, a_dag, a_dag_adj, b, b_adj, b_dag, b_dag_adj> = FreeAlgebra(QQ)

Pinv_a = pinv(a, a_dag, a_adj, a_dag_adj)
# b_dag is the Moore-Penrose inverse of the Gram matrix b
Pinv_b = pinv(...)
# define b to be the Gram matrix A^* A
b_def = ...

assumptions = add_adj(Pinv_a + Pinv_b + [b_def])
# state the claim A^dag = (A^* A)^dag A^*
claim = ...

Q = Quiver([(1, 2, u) for u in [a, a_dag_adj]] +
           [(2, 1, u) for u in [a_adj, a_dag]] +
           [(1, 1, u) for u in [b, b_adj, b_dag, b_dag_adj]])

proof = certify(assumptions, claim, Q)
print("The proof consists of %d steps." % len(proof))

## Exercise 7: The Moore-Penrose inverse of a real matrix is real
*(Fact 2 in [Hog13, Sec. I.5.7])*

If $A$ is a real matrix, then $A^\dagger$ is real as well.

*Needed:* the decomposition of the adjoint into conjugation and transposition (point 5). Here `b` denotes $A^\dagger$, and `a_tr`, `a_c` denote $A^T$ and $A^C$. The transposed and conjugated Moore-Penrose identities are given. You have to say that $A$ is real, which amounts to two polynomials: $A = A^C$, and — as a consequence — $A^T = A^\ast$.

In [ ]:
F.<a, a_tr, a_c, a_adj, b, b_tr, b_c, b_adj> = FreeAlgebra(QQ)

# the classical Moore-Penrose identities for b = A^dag, plus their adjoints
Pinv_b = add_adj(pinv(a, b, a_adj, b_adj))
# the transposed identities
Pinv_b_tr = [a_tr*b_tr*a_tr - a_tr, b_tr*a_tr*b_tr - b_tr,
             a_tr*b_tr - b_c*a_c, b_tr*a_tr - a_c*b_c]
# the conjugated identities
Pinv_b_c = [a_c*b_c*a_c - a_c, b_c*a_c*b_c - b_c]

# encode that a is real
a_real = ...

assumptions = Pinv_b + Pinv_b_tr + Pinv_b_c + a_real
# state the claim "A^dag is real"
claim = ...

Q = Quiver([(1, 2, u) for u in [a, a_c, b_tr, b_adj]] +
           [(2, 1, u) for u in [a_tr, a_adj, b, b_c]])

proof = certify(assumptions, claim, Q)

print("The proof is:")
pretty_print_proof(proof, assumptions)

## Exercise 8: Full rank decomposition
*(Fact 3 in [Hog13, Sec. I.5.7])*

If $A = BC$ is a full rank decomposition, i.e., $B$ has full column rank and $C$ has full row rank, then
$$A^\dagger = C^\ast(B^\ast A C^\ast)^{-1}B^\ast.$$

*Needed:* one-sided inverses (point 4) and identity matrices (point 3). Use `u` for a left inverse of `b`, `v` for a right inverse of `c`, `i` for the identity matrix, and `inv` for the inverse of $B^\ast AC^\ast$.

Note that $B^\ast AC^\ast = B^\ast BCC^\ast$ is indeed invertible, because $B^\ast B$ and $CC^\ast$ are.

In [ ]:
F.<a, b, c, i, u, v, x, a_adj, b_adj, c_adj, i_adj, u_adj, v_adj, x_adj, inv, inv_adj> = FreeAlgebra(QQ)

# x is the Moore-Penrose inverse of a
Pinv_x = pinv(a, x, a_adj, x_adj)
# the decomposition A = BC
a_decomposition = [a - b*c]
# b has full column rank (u is a left inverse of b) and
# c has full row rank (v is a right inverse of c)
full_rank = ...
# inv is a two-sided inverse of b_adj*a*c_adj
inverse = ...
# the identities satisfied by the identity matrix i
identity_matrix = [i*i - i, i - i_adj, b*i - b, i*c - c,
                   i*u - u, v*i - v, inv*i - inv, i*inv - inv]

assumptions = add_adj(Pinv_x + a_decomposition + full_rank + inverse + identity_matrix)
# state the claim A^dag = C^* (B^* A C^*)^(-1) B^*
claim = ...

Q = Quiver([(1, 2, w) for w in [a, x_adj]] +
           [(2, 1, w) for w in [a_adj, x]] +
           [(1, 3, w) for w in [c, v_adj]] +
           [(3, 1, w) for w in [c_adj, v]] +
           [(2, 3, w) for w in [b_adj, u]] +
           [(3, 2, w) for w in [b, u_adj]] +
           [(3, 3, w) for w in [i, i_adj, inv, inv_adj]])

proof = certify(assumptions, claim, Q)
print("The proof consists of %d steps." % len(proof))

## Exercise 9: Projections onto ranges
*(Fact 20 in [Hog13, Sec. I.5.7])*

$AA^\dagger$ is the orthogonal projection onto $\operatorname{range}(A)$. Part of this statement is the equality
$$\operatorname{range}(AA^\dagger) = \operatorname{range}(A),$$
which by Douglas' lemma amounts to two factorisations.

*Needed:* ranges (point 7). One of the two inclusions is done for you; find the factorisation that proves the other one.

In [ ]:
F.<a, a_adj, a_dag, a_dag_adj> = FreeAlgebra(QQ)

I = NCIdeal(add_adj(pinv(a, a_dag, a_adj, a_dag_adj)))

# range(A A^dag) is contained in range(A), because A A^dag = A X for some X
print(I.find_equivalent_expression(a*a_dag, prefix=a, heuristic='naive')[0])

# now the other inclusion: range(A) is contained in range(A A^dag)
print(I.find_equivalent_expression(..., prefix=..., heuristic='naive')[0])

## Exercise 10: The reverse order law
*(Fact 25 in [Hog13, Sec. I.5.7])*

In general $(AB)^\dagger \neq B^\dagger A^\dagger$. Several equivalent conditions are known that make the reverse order law hold. One of them is

$$(3) \qquad A^\dagger ABB^\ast A^\ast = BB^\ast A^\ast \quad\text{and}\quad BB^\dagger A^\ast AB = A^\ast AB.$$

Show that condition (3) implies $(AB)^\dagger = B^\dagger A^\dagger$.

*Needed:* Moore-Penrose identities for the *product* $AB$. Three spaces are involved: $B$ maps space 1 to space 2 and $A$ maps space 2 to space 3, so $(AB)^\dagger$ maps space 3 back to space 1.

This proof needs more than the default 10 iterations, so pass `maxiter=20` to `certify`.

In [ ]:
F.<a, b, a_dag, b_dag, ab_dag, a_adj, b_adj, a_dag_adj, b_dag_adj, ab_dag_adj> = FreeAlgebra(QQ)

Q = Quiver([(1, 2, w) for w in [b, b_dag_adj]] +
           [(2, 1, w) for w in [b_adj, b_dag]] +
           [(2, 3, w) for w in [a, a_dag_adj]] +
           [(3, 2, w) for w in [a_adj, a_dag]] +
           [(1, 3, ab_dag_adj), (3, 1, ab_dag)])

Pinv_a = pinv(a, a_dag, a_adj, a_dag_adj)
Pinv_b = pinv(b, b_dag, b_adj, b_dag_adj)
# ab_dag is the Moore-Penrose inverse of the product a*b
Pinv_ab = pinv(...)

basic_assumptions = Pinv_a + Pinv_b + Pinv_ab

# condition (3)
cond_3 = [a_dag*a*b*b_adj*a_adj - b*b_adj*a_adj,
          b*b_dag*a_adj*a*b - a_adj*a*b]
# the reverse order law
rol = [ab_dag - b_dag*a_dag]

# show that condition (3) implies the reverse order law
proofs = certify(...)
print("\nThe proofs consist of %s steps respectively." % str(list(map(len, proofs))))

# Part 4: When a statement is false

If `certify` cannot verify a claim, it reports

> `Failed! Not all ideal memberships could be verified.`

and returns `False`. This is *not* a proof that the claim is wrong: the search for a proof only runs for a bounded number of iterations, and increasing `maxiter` may well succeed.

To actually refute a statement, the package can search for a counterexample. The command `construct_counterexample` looks for concrete matrices that satisfy all assumptions but violate the claim. The sizes of these matrices are prescribed by a dictionary that assigns a dimension to every vertex of the quiver.

## An example: does $AA^\dagger = A^\dagger A$ hold?

Exercise 3 showed that a *normal* matrix commutes with its Moore-Penrose inverse. What about an arbitrary square matrix? Let us first ask `certify`.

In [ ]:
F.<a, a_adj, a_dag, a_dag_adj> = FreeAlgebra(QQ)

# A is square, so the quiver has a single vertex
Q = Quiver([(1, 1, u) for u in [a, a_adj, a_dag, a_dag_adj]])

assumptions = add_adj(pinv(a, a_dag, a_adj, a_dag_adj))
claim = a*a_dag - a_dag*a

proof = certify(assumptions, claim, Q)
print("\ncertify returned %s." % str(proof))

No proof was found. To see whether the statement is actually false, we look for a counterexample. The fourth argument assigns a dimension to every vertex of the quiver — here, a single vertex, for which we try $2 \times 2$ matrices.

In [ ]:
counterexample = construct_counterexample(assumptions, claim, Q, {'1': 2})
print_solution(counterexample, [a])

So
$$
A = \begin{pmatrix} 0 & 1 \\ 0 & 0 \end{pmatrix}
$$
should be a counterexample. We can check this result independently — here with SageMath's own `pseudoinverse` method.

In [ ]:
A = matrix(QQ, [[0, 1], [0, 0]])

print("A^dag =\n%s\n" % str(A.pseudoinverse()))
print("A*A^dag =\n%s\n" % str(A * A.pseudoinverse()))
print("A^dag*A =\n%s\n" % str(A.pseudoinverse() * A))

A word of warning: the search is only carried out for the prescribed dimensions. If no counterexample is found, one may still exist in a larger dimension — but the search space grows quickly, so raising the dimensions is not always feasible.

## Try yourself: the reverse order law without extra hypotheses

Exercise 10 showed that condition (3) implies $(AB)^\dagger = B^\dagger A^\dagger$. Convince yourself that some such condition is really needed, i.e., that the reverse order law does *not* hold in general:

1. set up the Moore-Penrose identities for $A$, $B$ and $AB$, and assume nothing else;
2. check that `certify` does not prove $(AB)^\dagger = B^\dagger A^\dagger$;
3. search for a counterexample, taking all three spaces to be $2$-dimensional;
4. verify the counterexample, as above, with SageMath's `pseudoinverse`.

In [ ]:
F.<a, b, a_dag, b_dag, ab_dag, a_adj, b_adj, a_dag_adj, b_dag_adj, ab_dag_adj> = FreeAlgebra(QQ)

Q = Quiver([(1, 2, w) for w in [b, b_dag_adj]] +
           [(2, 1, w) for w in [b_adj, b_dag]] +
           [(2, 3, w) for w in [a, a_dag_adj]] +
           [(3, 2, w) for w in [a_adj, a_dag]] +
           [(1, 3, ab_dag_adj), (3, 1, ab_dag)])

# the Moore-Penrose identities for a, b and a*b -- and nothing else
assumptions = ...
# the reverse order law
claim = ...

# 1. certify does not find a proof
proof = certify(assumptions, claim, Q)
print("\ncertify returned %s.\n" % str(proof))

# 2. every vertex of the quiver carries a 2-dimensional space
counterexample = construct_counterexample(...)

In [ ]:
# 3. verify the counterexample with SageMath's own pseudoinverse
A = matrix(QQ, ...)
B = matrix(QQ, ...)

print("(A*B)^dag =\n%s\n" % str((A*B).pseudoinverse()))
print("B^dag*A^dag =\n%s\n" % str(B.pseudoinverse() * A.pseudoinverse()))
print("Is (A*B)^dag = B^dag*A^dag? %s"
      % str((A*B).pseudoinverse() == B.pseudoinverse() * A.pseudoinverse()))

# Where to go from here

Have fun exploring and using the package! Look for other operator statements in the literature or try to (dis)prove your own research statements. If you have suggestions for improvements or you find any bugs, please let us know under [clemens.hofstadler@jku.at](mailto:clemens.hofstadler@jku.at).

# References

For further information on the approach and the underlying theory, see [BHR23]. The theorems in this notebook are taken from [Hog13].

[BHR23]&nbsp;&nbsp; Bernauer, K., Hofstadler, C., Regensburger, G. How to Automatise Proofs of Operator Statements: Moore–Penrose Inverse; A Case Study. In: Computer Algebra in Scientific Computing (CASC) 2023. (2023) https://doi.org/10.1007/978-3-031-41724-5_3



[Hog13]&nbsp;&nbsp;Hogben L., Handbook of Linear Algebra. CRC press, 2 edn. (2013)
